In [ ]:
import math

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import matplotlib.pyplot as plt
import numpy as np
import sympy as sym

from rotation.symbolic.general import ANGLE, MASS, COUPLING, create_3d_rotation
from rotation.symbolic.pseudo import create_mix_equation
from rotation.numerical.solver import MASS_NEUTRAL_PION, MASS_CHARGED_KAON, MASS_ETA, initialize_system, create_func, solve_mixing
from rotation.numerical.pseudo import pseudo_calculate_meson_couplings, pseudo_calculate_bare_masses, pseudo_calculate_photon_coupling, pseudo_calculate_bare_kaon, pseudo_calculate_kaon_couplings

In [ ]:
particles = tuple(["pi0", "phi", "eta"])

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [MASS[k] for k in ["pi0", "pi0Bare"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / MASS_NEUTRAL_PION
    elif key in [MASS[k] for k in ["eta", "etaBare"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / MASS_ETA
    elif key in [MASS[k] for k in ["phi", "phiBare", "pi0-phi", "eta-phi"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ANGLE.values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

## Prepare solver

In [ ]:
rotate = create_3d_rotation()[0]
equation = create_mix_equation(rotate)[0]
argument_order, system_of_equations = initialize_system(equation.lhs - equation.rhs)

## Solver inputs

In [ ]:
# Fixed known parameters
mphi = 2.5
quarks = 0.7, 0.1, -0.2
piphi, etaphi = pseudo_calculate_bare_masses(
    coupling_up=quarks[0],
    coupling_down=quarks[1],
    coupling_strange=quarks[2],
)
known = {
    MASS["pi0"]: MASS_NEUTRAL_PION,
    MASS["eta"]: MASS_ETA,
    MASS["phiBare"]: mphi,
    MASS["pi0-phi"]: piphi,
    MASS["eta-phi"]: etaphi,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    MASS["phi"]: mphi,
    MASS["pi0Bare"]: MASS_NEUTRAL_PION,
    MASS["etaBare"]: MASS_ETA,
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

# Printout the system with our known values, and unknowns blank
print("Arguments input:")
for k in argument_order:
    if k in known:
        sym.Eq(k, known[k], evaluate=False)
    else:
        sym.Eq(k, sym.Symbol("?"), evaluate=False)
equation.subs(known)

## Run solver

In [ ]:
# Create function
func = create_func(
    system_of_equations,
    argument_order,
    known,
)
try:
    # Solve system
    solved = solve_mixing(
        func,
        argument_order,
        unknown,
    )
    print("Solver succeeded!")
except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")

## Solution display

In [ ]:
# Printout the system with our solved unknowns, and knowns blank
print("Arguments solved:")
for k, v in solved.items():
    v = format_output(k, v)
    sym.Eq(k, v, evaluate=False)

print("The solved equation:")
equation.subs(solved)

print("Compare LHS and RHS evaluations:")
sym.Eq(
    (1*equation.lhs.subs(known|solved)).evalf(),
    (1*equation.rhs.subs(known|solved)).evalf(),
    evaluate=False
)

## Calculate Couplings

In [ ]:
gchi = 0.2
meson_couplings = pseudo_calculate_meson_couplings(
    bare_mediator_dm_coupling = gchi,
    alpha = solved[ANGLE["alpha"]],
    beta = solved[ANGLE["beta"]],
    gamma = solved[ANGLE["gamma"]],
)
for s,g in zip([COUPLING[f"{k}-chi"] for k in particles], meson_couplings):
    sym.Eq(s,g,evaluate=False)

In [ ]:
photon_coupling = pseudo_calculate_photon_coupling(
    alpha = solved[ANGLE["alpha"]],
    beta = solved[ANGLE["beta"]],
    gamma = solved[ANGLE["gamma"]],
)
sym.Eq(COUPLING["phi-gamma"], photon_coupling, evaluate=False)

In [ ]:
bare_kaon_couplings = pseudo_calculate_bare_kaon(energy_pion = 1)
for s,g in zip([COUPLING[f"{k}Bare-pi-K"] for k in particles], bare_kaon_couplings):
    sym.Eq(s,g,evaluate=False)

print("After rotating:")
kaon_couplings = pseudo_calculate_kaon_couplings(
    bare_pion_pion_kaon_coupling = bare_kaon_couplings[0],
    bare_mediator_pion_kaon_coupling = bare_kaon_couplings[1],
    bare_eta_pion_kaon_coupling = bare_kaon_couplings[2],
    alpha = solved[ANGLE["alpha"]],
    beta = solved[ANGLE["beta"]],
    gamma = solved[ANGLE["gamma"]],
)
for s,g in zip([COUPLING[f"{k}-pi-K"] for k in particles], kaon_couplings):
    sym.Eq(s,g,evaluate=False)

In [ ]:
# Creating the kaon couplings as a function of the pion energy
def kaon_function(energy):
    return pseudo_calculate_kaon_couplings(
        *pseudo_calculate_bare_kaon(energy),
        *[solved[ANGLE[k]] for k in ["alpha", "beta", "gamma"]],
    )

xs = np.linspace(0,10,num=101)
fig, ax = plt.subplots(1,1)
for s, ys in zip([COUPLING[f"{k}-pi-K"] for k in particles], kaon_function(xs)):
    _ = ax.plot(xs, ys, label=f"${s}$")
_ = ax.legend()

## Loop the Solver

In [ ]:
x_label = MASS["phiBare"]
x_min, x_max = 0, 5

solutions = []
for x in np.linspace(x_min, x_max,  num=101):
    known[x_label] = x
    func = create_func(
        system_of_equations,
        argument_order,
        known,
    )
    try:
        solved = solve_mixing(
            func,
            argument_order,
            unknown,
        )
        solutions.append([x, known.copy(), solved.copy()])
    except ValueError as e:
        print(f"{x = :.3e}", end="\t")
        print(e)

In [ ]:
xs = np.array([sol[0] for sol in solutions])
# Reorganize solutions to be arrays of each unknown
data = {}
for sol in solutions:
    for y_label, data_point in sol[2].items():
        try:
            data[y_label].append(data_point)
        except KeyError:
            data[y_label] = [data_point]
for y_label, ys in data.items():
    data[y_label] = np.array(ys)
    data[y_label] = np.ma.array(ys, mask=data[y_label] < 0.0)

if not data:
    print("No data!")
else:
    # plotting
    fig, axs = plt.subplots(1,2, figsize=(10,6))
    mass_selector = {MASS[k]: v for k,v in zip(["pi0Bare", "etaBare", "phi"], [MASS_NEUTRAL_PION, MASS_ETA, mphi])}
    
    for y_label, ys in data.items():
        if y_label in ANGLE.values():
            ax = axs[1]
            y_data = ys.data
        else:
            ax = axs[0]
            y_data = ys.data / mass_selector[y_label]
            if y_label == MASS["phi"]:
                y_data = 1/y_data
        ax.scatter(xs, y_data, label=f"${y_label}$", marker=".")
    # axs[0].set_ylabel("Mass [GeV]")
    axs[0].set_ylabel("Bare Mass / Rotated Mass")
    axs[1].set_ylabel("Angle")
    fig.suptitle(f"${COUPLING['u']} = {quarks[0]}$, ${COUPLING['d']} = {quarks[1]}$, ${COUPLING['s']} = {quarks[2]}$")
    for ax in axs.flatten():
        ax.set_xlim(x_min, x_max)
        ax.set_xlabel(f"${x_label}$")
        ax.legend()
        ax.grid(True)
    fig.tight_layout()
    fig.show()